In [1]:
%pip install -r ../../requirements-offline.txt --quiet

/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-31/.check-env/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Exercise — Module M6: PatchCore anomaly detection

**วิธีทำ:** เติมช่อง `# TODO` ให้ทุกเซลล์รันผ่าน แล้วตอบคำถามในช่อง **คำตอบ** — รัน offline บน CPU เหมือน book ทุกประการ (เซลล์เตรียมข้อมูล + เทรนให้มาแล้ว รันผ่านก่อนแล้วใช้ต่อ)

**เป้าหมายหลักของแล็บนี้:** ปลายทางจริงคือการใช้ **รูปถ่ายของคุณเอง** — ข้อ 1 คือคู่มือถ่ายรูป + เตรียมข้อมูล

## 0. เตรียมข้อมูล + เทรนโมเดล (ให้มาแล้ว — รันผ่าน)

โค้ดชุดนี้คือส่วนสำคัญของ book ย่อมาไว้ด้วยกัน: สร้างภาพสังเคราะห์ (แผ่นโลหะ 20 ปกติ + 8 ชำรุด) → เทรน PatchCore → ตัวช่วย `score_paths()` และผลคะแนน ไว้ใช้ทุกข้อ

In [2]:
import logging  # noqa: E402
import warnings  # noqa: E402

warnings.filterwarnings("ignore")  # ตัด warning รบกวนจาก libs
logging.getLogger("lightning").setLevel(logging.ERROR)  # และ log ของ Lightning

from pathlib import Path  # noqa: E402

import numpy as np  # noqa: E402
from PIL import Image, ImageDraw  # noqa: E402

SEED = 42


def repo_root() -> Path:
    'หา repo root โดยไล่ขึ้นจาก cwd.'
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise FileNotFoundError("ไม่เจอ repo root — เปิดโน้ตบุ๊กจากใน repo นี้")


ROOT = repo_root()

# backbone ของ anomalib (timm) — ใช้ datasets/models/hf ที่
# pre-bundle ไว้ถ้ามี; ถ้าไม่มี ใช้ default cache ตามเดิม
import os  # noqa: E402

_hf = ROOT / "datasets" / "models" / "hf"
if _hf.exists():
    os.environ["HF_HOME"] = str(_hf)
SAMPLE_DIR = ROOT / "data" / "m6_sample"
RUN_DIR = ROOT / "data" / "m6_results"


def metal_plate(seed: int, w: int = 256, h: int = 256) -> Image.Image:
    'วาดแผ่นโลหะ "ปกติ": ลายเส้นผิวขัด + grain + แสงตกต่างกันเล็กน้อย.'
    r = np.random.default_rng(seed)
    gain = 1.0 + r.uniform(-0.06, 0.06)
    base = np.full((h, w, 3), 100.0)
    stripes = np.tile(np.arange(h), (w, 1)).T % 10 < 4
    base[stripes] -= 12
    noise = r.normal(0, 10, (h, w, 1))
    img = np.clip((base + noise) * gain, 0, 255).astype(np.uint8)
    for sl in (img[:8], img[-8:], img[:, :8], img[:, -8:]):
        sl[:] = np.clip(sl - 45, 0, 255)
    return Image.fromarray(img)


def add_defect(img: Image.Image, kind: int, i: int) -> Image.Image:
    'วาดความชำรุด 4 แบบลงบนแผ่นโลหะ: ขีดข่วน/กัด/ไหม้/สนิม.'
    d = ImageDraw.Draw(img)
    if kind == 0:      # รอยขีดข่วนลึก
        x0 = 50 + (i * 9) % 110
        d.line([(x0, 45), (x0 + 18, 210)], fill=(10, 8, 6), width=9)
    elif kind == 1:    # รอยกัด/รูสึก
        cx, cy = 60 + (i * 11) % 100, 55 + (i * 13) % 100
        d.ellipse([cx, cy, cx + 26, cy + 26], fill=(240, 238, 232))
        d.ellipse([cx + 4, cy + 4, cx + 18, cy + 18], fill=(30, 25, 20))
    elif kind == 2:    # คราบไหม้
        cx, cy = 45 + (i * 17) % 110, 45 + (i * 19) % 100
        d.ellipse([cx, cy, cx + 52, cy + 44], fill=(28, 22, 16))
    else:              # สนิมกระจาย
        cx, cy = 55 + (i * 23) % 100, 50 + (i * 7) % 105
        d.ellipse([cx, cy, cx + 44, cy + 36], fill=(150, 70, 30))
        rr = np.random.default_rng(9000 + i)
        for _ in range(14):
            px = cx + int(rr.integers(2, 36))
            py = cy + int(rr.integers(2, 28))
            s = int(rr.integers(3, 7))
            d.ellipse([px, py, px + s, py + s], fill=(25, 15, 8))
    return img

In [3]:
import shutil  # noqa: E402

N_GOOD, N_DEFECT = 20, 8
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
(SAMPLE_DIR / "good").mkdir(parents=True)
(SAMPLE_DIR / "defect").mkdir(parents=True)
for i in range(N_GOOD):
    metal_plate(1000 + i).save(SAMPLE_DIR / "good" / f"normal_{i:02d}.png")
for i in range(N_DEFECT):
    img = add_defect(metal_plate(3000 + i), kind=i % 4, i=i)
    img.save(SAMPLE_DIR / "defect" / f"defect_{i:02d}.png")
print("สร้างภาพแล้ว:", N_GOOD, "ปกติ +", N_DEFECT, "ชำรุด")

สร้างภาพแล้ว: 20 ปกติ + 8 ชำรุด


ต่อ: เทรน PatchCore (จบในหลักวินาที — ไม่มี gradient จริง ๆ ตามที่เรียนใน book)

In [4]:
import torch  # noqa: E402
from anomalib.data import Folder  # noqa: E402
from anomalib.engine import Engine  # noqa: E402
from anomalib.models import Patchcore  # noqa: E402

torch.manual_seed(SEED)
datamodule = Folder(
    name="m6_sample",
    root=str(SAMPLE_DIR),
    normal_dir="good",
    abnormal_dir="defect",
    val_split_mode="from_test",
    val_split_ratio=0.3,
    num_workers=0,
    seed=SEED,
)
model = Patchcore(
    layers=("layer2", "layer3"),
    num_neighbors=3,
    post_processor=False,
    visualizer=False,
)
engine = Engine(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    default_root_dir=str(RUN_DIR),
    enable_progress_bar=False,
)
engine.fit(model=model, datamodule=datamodule)
print("เทรนเสร็จ")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


20 hidden files found in /Users/utarn/projects/ml-metrology/.claude/worktrees/issue-31/data/m6_sample/good. Please make sure this is intended.


8 hidden files found in /Users/utarn/projects/ml-metrology/.claude/worktrees/issue-31/data/m6_sample/defect. Please make sure this is intended.


...

Selecting Coreset Indices.:  88%|████████▊ | 1446/1637 [00:01<00:00, 1324.23it/s]

Selecting Coreset Indices.:  97%|█████████▋| 1591/1637 [00:01<00:00, 1360.76it/s]

Selecting Coreset Indices.: 100%|██████████| 1637/1637 [00:01<00:00, 1314.55it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


เทรนเสร็จ


In [5]:
from torchvision.transforms.v2.functional import to_dtype  # noqa: E402
from torchvision.transforms.v2.functional import to_image  # noqa: E402


def prep(path: str) -> torch.Tensor:
    'โหลดภาพ 1 ใบ → tensor ที่โมเดลคาดหวัง (256×256, normalized).'
    img = to_image(Image.open(path).convert("RGB"))
    return model.pre_processor(to_dtype(img, torch.float32, scale=True))


def score_paths(paths: list[str], batch_size: int = 8):
    'ให้คะแนนภาพหลายใบ → (scores, anomaly maps ขนาด 256×256 ต่อใบ).'
    scores, maps = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(paths), batch_size):
            batch = torch.stack([prep(p) for p in paths[i:i + batch_size]])
            out = model(batch)
            scores.append(out.pred_score)
            maps.append(out.anomaly_map.squeeze(1))
    return torch.cat(scores).numpy(), torch.cat(maps).numpy()


good_paths = sorted((SAMPLE_DIR / "good").glob("*.png"))
defect_paths = sorted((SAMPLE_DIR / "defect").glob("*.png"))
s_good, _ = score_paths([str(p) for p in good_paths])
s_def, maps_def = score_paths([str(p) for p in defect_paths])
print("คะแนนภาพปกติ :", np.round(s_good, 0))
print("คะแนนภาพชำรุด:", np.round(s_def, 0))

คะแนนภาพปกติ : [182. 132. 189. 178. 181. 190. 200. 180. 165. 180. 180. 199. 161. 169.
 159. 177. 176. 152. 152. 183.]
คะแนนภาพชำรุด: [249. 245. 501. 247. 254. 216. 496. 243.]


## ข้อ 1 — ถ่ายรูปของคุณเอง แล้วเทรนด้วยมัน (งานหลักของแล็บ)

**คู่มือถ่ายรูป — ทำตามนี้แล้ว PatchCore จะใช้งานได้จริง:**

1. **เลือกวัตถุที่มี "หน้าตาปกติ" ซ้ำ ๆ ได้** — เครื่องมือวัด, ชิ้นงาน, ปุ่ม/ผิวเครื่องจักร, ป้าย, ผิวเอกสาร
2. **วางวัตถุบนพื้นหลังเรียบสีเดิมทุกครั้ง** (กระดาษขาว/ผ้าสีเรียบ) — ใช้กรอบ/กล่องช่วยกำหนดตำแหน่งก็ได้
3. **แสงคงที่** — ห้องเดียวกัน ไฟชุดเดียวกัน หลีกเลี่ยงแสงแดด/เงาคนถ่าย
4. **มุม + ระยะคงที่** — พยุงมือถือ/กล้องกับโต๊ะ หรือใช้ขาตั้ง; ภาพทุกใบต้องมีวัตถุเต็มเฟรมแบบเดียวกัน
5. ถ่าย **15–20 ใบ "ปกติ"** (เปลี่ยนตำแหน่งเล็กน้อยได้ แต่ไม่เกิน ±2 ซม.) + **3–5 ใบ "ชำรุด"** (ตั้งใจทำตำหนิ: ขีด, ติดฝุ่น, รอยช้ำ, สนิม — อะไรก็ได้ที่งานจริงควรจับ)
6. สร้างโฟลเดอร์ `data/my_sample/good/` กับ `data/my_sample/defect/` แล้วลากไฟล์เข้าไป

**TODO 1:** ชี้ `MY_DIR` ไปที่โฟลเดอร์ของคุณ (ถ้ายังไม่ได้ถ่าย — เซลล์นี้จะ fallback ไปใช้ชุดตัวอย่างสังเคราะห์ให้ก่อน เพื่อให้เห็นผลลัพธ์)

In [6]:
MY_DIR = ROOT / "data" / "my_sample"

if (MY_DIR / "good").is_dir() and (MY_DIR / "defect").is_dir():
    my_good = sorted((MY_DIR / "good").glob("*"))
    my_defect = sorted((MY_DIR / "defect").glob("*"))
    print("ใช้รูปของคุณ:", len(my_good), "ปกติ +", len(my_defect), "ชำรุด")
else:
    print("ยังไม่มี data/my_sample/ — ใช้ชุดตัวอย่างสังเคราะห์แทนก่อน")
    my_good, my_defect = good_paths, defect_paths

# TODO 1: เทรนโมเดลใหม่จากรูปของคุณ — คัดลอกชุดคำสั่งเทรนจากเซลล์ด้านบน
# เปลี่ยน root เป็น str(MY_DIR) เปลี่ยน name เป็น "my_sample"
# แล้ว engine.fit() ใหม่ (อย่าลืม torch.manual_seed(SEED) ก่อน)


ยังไม่มี data/my_sample/ — ใช้ชุดตัวอย่างสังเคราะห์แทนก่อน


**คำตอบข้อ 1:** _(เติมตรงนี้ — รูปของคุณ AUC ได้เท่าไร ขั้นถ่ายรูปขั้นไหนพลาดบ่อย เช่น แสงเปลี่ยนระหว่างถ่าย, ตำแหน่งวัตถุขยับ, พื้นหลังไม่เรียบ)_

## ข้อ 2 — เล่นกับพารามิเตอร์ของ PatchCore

พารามิเตอร์สองตัวนี้คือก้าวเร่ง/เบรกของ PatchCore:

- `num_neighbors` (k ของ kNN) — คะแนน patch = ระยะไปเพื่อนบ้านอันดับ k
- `coreset_sampling_ratio` (default 0.1) — สัดส่วน patch ที่เก็บไว้ใน memory bank

**TODO 2:** เทรนโมเดลใหม่ 2 รอบ — (a) `num_neighbors=1`, (b) `num_neighbors=9` — แล้วเทียบ AUC กับค่ารอบ `num_neighbors=3` ด้านบน (คำใบ้: สร้าง `Patchcore(...)` ใหม่ ใส่ `post_processor=False, visualizer=False` เหมือนเดิม **และสร้าง `Engine` ใหม่ด้วย** (Engine เก่าที่เคย fit แล้วจะไม่เทรนโมเดลใหม่) แล้ว `engine.fit` + `score_paths` อีกรอบ อย่าลืม `torch.manual_seed(SEED)` ก่อนแต่ละรอบ)

In [7]:
from sklearn.metrics import roc_auc_score  # noqa: E402

y_true = [0] * len(s_good) + [1] * len(s_def)
y_score = np.concatenate([s_good, s_def])
print(f"num_neighbors=3 → AUC = {roc_auc_score(y_true, y_score):.3f}")

# TODO 2: วนลูป num_neighbors = [1, 9]
#   - สร้าง Patchcore ใหม่ (post_processor=False, visualizer=False เหมือนเดิม)
#   - สร้าง Engine ใหม่ด้วย (Engine เก่าที่เคย fit แล้วจะไม่ยอมเทรนโมเดลใหม่)
#   - engine.fit ใหม่ แล้ว score_paths
#   - พิมพ์ AUC ของแต่ละค่า


num_neighbors=3 → AUC = 1.000


**คำตอบข้อ 2:** _(เติมตรงนี้ — k=1 / k=9 ต่างจาก k=3 อย่างไร ทำไม k มาก ๆ จึง "ใจดี" ขึ้นกับ patch ที่พรุนเล็กน้อย แต่แลกกับความไว)_

## ข้อ 3 — เลือก threshold แบบนักเมโทรโลยี

**TODO 3:** กำหนด threshold จากคะแนนภาพปกติด้วย `np.percentile(s_good, Q)` — วนลูป Q = 50, 80, 95, 99 สำหรับแต่ละ Q นับ **FP** (ปกติที่ score > threshold) และ **FN** (ชำรุดที่ score <= threshold)

**คำถาม:** สมมติต้นทุน "ตรวจพลาด defect จริง 1 ชิ้น" = สูงกว่า "ปลอมเตือน 1 ครั้ง" 10 เท่า — Q ไหนที่คุณจะเลือกใช้จริง เพราะอะไร

In [8]:
# TODO 3: วนลูป Q = [50, 80, 95, 99]
#   - THRESHOLD = np.percentile(s_good, Q)
#   - FP = จำนวน s_good ที่ > THRESHOLD, FN = จำนวน s_def ที่ <= THRESHOLD
#   - พิมพ์ Q, THRESHOLD, FP, FN ต่อแถว


**คำตอบข้อ 3:** _(เติมตรงนี้ — Q ที่เลือก + เหตุผลเชิงต้นทุน)_

## ข้อ 4 — อ่าน heatmap ให้เป็น

**TODO 4:** เลือกภาพชำรุดที่ได้คะแนน**ต่ำสุด** (`np.argmin(s_def)`) แล้ววาดภาพจริง + anomaly map ซ้อนทับ (คำใบ้: คัดลอกโค้ด heatmap จาก book หัวข้อ 6 เปลี่ยน index เป็นค่าที่ได้)

**คำถาม:** จุดสว่างใน heatmap ตรงกับตำแหน่ง defect จริงไหม — ถ้าไม่ตรง เดาว่าอะไรทำให้โมเดลสนใจจุดแปลกตรงนั้น (เช่น grain ที่มืดกว่าปกติ, ขอบภาพ)

In [9]:
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = ["DejaVu Sans", "Noto Sans Thai", "Tahoma"]

# TODO 4: i_low = int(np.argmin(s_def))
#   วาด 3 ช่อง: ภาพจริง / anomaly map (cmap="inferno") / ซ้อนทับ (alpha)


**คำตอบข้อ 4:** _(เติมตรงนี้ — heatmap ตรงจุดชำรุดจริงไหม สิ่งอื่นที่โมเดลมองว่าแปลก)_

## ข้อ 5 — ภาพปกติที่ "ถ่ายใหม่": false positive เกิดจากอะไร

PatchCore ไวต่อสิ่งที่ไม่เคยเห็น — รวมถึง**การเปลี่ยนมุม/แสง/พื้นหลัง** ของวัตถุเดิม

**TODO 5:** สร้างภาพปกติ "ถ่ายใหม่" 2 แบบ: (a) `metal_plate(7777)` — สภาพเดิมทุกอย่างแค่ seed ใหม่ (เหมือนถ่ายวัตถุเดิมมุมเดิม), (b) ภาพเดียวกัน**หมุน 8 องศา** (`metal_plate(7777).rotate(8)`) — แล้ว `score_paths` ทั้งสองใบ เทียบกับ threshold จากข้อ 3 (Q=95)

**คำถาม:** ใบไหนโดนตอบ "ชำรุด" บ้าง เพราะอะไร — และเชื่อมโยงกับการถ่ายรูปของคุณเอง (ข้อ 1): กฎข้อไหนในคู่มือถ่ายรูปช่วยกันปัญหานี้

In [10]:
plate_same = metal_plate(7777)           # (a) สภาพเดิม
plate_rot = metal_plate(7777).rotate(8)  # (b) หมุน 8 องศา
q_dir = SAMPLE_DIR.parent / "m6_unseen"
q_dir.mkdir(exist_ok=True)
plate_same.save(q_dir / "q5_same.png")
plate_rot.save(q_dir / "q5_rot.png")

# TODO 5: score_paths สองไฟล์นี้ แล้วเทียบกับ THRESHOLD (Q=95)
#   พิมพ์ score แต่ละใบ + สรุปว่า threshold ตอบ "ชำรุด" หรือ "ปกติ"


**คำตอบข้อ 5:** _(เติมตรงนี้ — ใบหมุนโดนไหม เพราะ patch ของภาพหมุนหาเพื่อนบ้านไม่เจอ; กฎถ่ายรูปข้อไหนกันเรื่องนี้)_

## สรุป (เขียนสั้น ๆ จากสิ่งที่ทำมา)

- สิ่งที่ surprise คุณที่สุดจากแล็บนี้ (เช่น ไม่ต้องมี label, เทรนเร็วมาก, heatmap ชี้จุดได้):
- ถ้าจะใช้กับงานจริงของคุณ ปัญหาแรกที่คาดว่าจะเจอคือ: _(เติมตรงนี้)_

เมื่อเสร็จแล้ว เทียบกับ `solution.ipynb` — และอ่านเชิงลึกต่อใน `explanation.md`